# 环节 03 · 矩阵的秩与张成空间（配套 Notebook）

> 配套长文：[环节03-矩阵的秩与张成空间详解.md](./环节03-矩阵的秩与张成空间详解.md) · 导航：[环节00](./环节00-总揽与环节导航.md)
> 定位：外积还原「幸运矩阵」、2×2 张成直线 vs 平面、幂迭代找最近的秩 1 表。**纯标准库。**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 幸运 3×3 | §2 | 9 个数 = 外积 3+3 |
| §2 2×2 张成 | §3 | 共线走不出直线 |
| §3 秩 2 依赖 | §3 | 第三行 = 前两行之和 |
| §4 幂迭代 | §4 | 满秩表仍可压成一句主信息 |


In [ ]:
def fmt(mat, nd=1):
    return "\n".join("  [" + ", ".join(f"{v:{nd+4}.{nd}f}" for v in row) + "]" for row in mat)

def outer(u, v):
    return [[ui * vj for vj in v] for ui in u]

def matvec(m, x):
    return [sum(mi * xj for mi, xj in zip(row, x)) for row in m]

def transpose(m):
    return [list(col) for col in zip(*m)]

def add(a, b):
    return [[x + y for x, y in zip(ra, rb)] for ra, rb in zip(a, b)]

def scale(m, g):
    return [[g * x for x in row] for row in m]

def frobenius(m):
    return sum(x * x for row in m for x in row) ** 0.5

def norm(x):
    return sum(t * t for t in x) ** 0.5


## 1. 幸运矩阵：三句话其实是一句（长文 §2）

`[[1,2,3],[10,20,30],[100,200,300]]` = `[1,10,100]ᵀ · [1,2,3]`。


In [ ]:
lucky = [
    [1.0, 2.0, 3.0],
    [10.0, 20.0, 30.0],
    [100.0, 200.0, 300.0],
]
u = [1.0, 10.0, 100.0]
v = [1.0, 2.0, 3.0]
recon = outer(u, v)

print("原矩阵（9 个数）：\n" + fmt(lucky))
print("\n外积还原（3+3 个数）：\n" + fmt(recon))
print(f"\n逐格相等？ {recon == lucky}")
print("第二行 = 第一行 × 10，第三行 = 第一行 × 100 → 秩 1。")
print("传这张表：寄一行 + 一列倍数，不必寄九个数。")


## 2. 张成：共线走不出直线（长文 §3）

目标点 (4, 1) 不在 y=2x 上。秩 1 的两支箭头怎么走都到不了；秩 2 可以。


In [ ]:
rank2 = [[1.0, 2.0], [3.0, 4.0]]   # 行：(1,2), (3,4)
rank1 = [[1.0, 2.0], [2.0, 4.0]]   # 行：(1,2), (2,4) 共线
target = [4.0, 1.0]

def det2(m):
    return m[0][0] * m[1][1] - m[0][1] * m[1][0]

def solve2(m, t):
    """α·row0 + β·row1 = t。m 按行堆两个向量。"""
    a11, a12 = m[0][0], m[1][0]   # 第一列 = 两个向量的 x
    a21, a22 = m[0][1], m[1][1]   # 第二列 = 两个向量的 y
    den = a11 * a22 - a12 * a21
    if abs(den) < 1e-12:
        return None
    alpha = (t[0] * a22 - a12 * t[1]) / den
    beta = (a11 * t[1] - t[0] * a21) / den
    return alpha, beta

print(f"秩 2 矩阵 det={det2(rank2):.0f}（非 0 → 张成整个平面）")
print(f"秩 1 矩阵 det={det2(rank1):.0f}（=0 → 张成一条直线）\n")

sol = solve2(rank2, target)
print(f"目标点 {target}")
print(f"  秩 2：α·(1,2) + β·(3,4) → α={sol[0]:.1f}, β={sol[1]:.1f}  （允许倒走）")
print(f"  秩 1：无解，两支箭头都在 y=2x 上，到不了 (4,1)。")


## 3. 秩 2：第三行是前两行的和（长文 §3）

依赖不必是「整行乘一个数」。线性组合也算同一平面。


In [ ]:
r0 = [1.0, 2.0, 3.0]
r1 = [10.0, 10.0, 10.0]
r2 = [r0[i] + r1[i] for i in range(3)]
rank2_3x3 = [r0, r1, r2]
print("3×3 秩 2：\n" + fmt(rank2_3x3))
print(f"\n第三行 == 第一行 + 第二行？ {r2 == [11.0, 12.0, 13.0]}")
print("三支箭头落在同一平面，张不成 3D 体积 → 秩 2。")
print("写成 3×2 × 2×3 就够，不必 3×3。")


## 4. 满秩也能逼近：幂迭代找最近的秩 1（长文 §4）

标准 LoRA **不对 W 做 SVD**，只是「低秩能贴满秩」这句话需要一个可跑的例子。


In [ ]:
def rank1_approx(m, steps=40):
    """幂迭代：最近的秩 1 矩阵 σ u vᵀ（Frobenius）。"""
    n = len(m[0])
    v = [1.0] * n
    mt = transpose(m)
    for _ in range(steps):
        u = matvec(m, v)
        nu = norm(u) or 1.0
        u = [t / nu for t in u]
        v = matvec(mt, u)
        nv = norm(v) or 1.0
        v = [t / nv for t in v]
    sigma = sum(ui * xi for ui, xi in zip(u, matvec(m, v)))
    approx = scale(outer(u, v), sigma)
    return sigma, u, v, approx

full = [
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 10.0],   # 故意不满秩 1，也不等于前两行组合
]
sigma, u, v, approx = rank1_approx(full)
resid = add(full, scale(approx, -1.0))

print("原矩阵（满秩味道）：\n" + fmt(full))
print(f"\n最近秩 1（σ={sigma:.3f}）：\n" + fmt(approx, nd=2))
print("\n残差：\n" + fmt(resid, nd=2))
print(f"\n||原||={frobenius(full):.3f}  ||残差||={frobenius(resid):.3f}")
print("主信息被一句外积拿走了；剩下的才是「第二、第三句话」。")
print("LoRA 的先验：好的 ΔW 几乎只需要前几句话（低秩）。")
